In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
db_path = Path(r"D:\BDDPlabacomCoordinador")
df_be = pd.read_parquet(Path(r"D:\ProyectoAnalisisElectrico\BarrasEstaciones\BarrasEstaciones.parquet")) 


In [3]:
lista_locs = []
lista_nolocs = []

for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    origin, date, version = folder_date.name.split("_")
    print(f"Mirando la fecha  {date}")
    
    target_dir = Path(r"D:\BDDPlabacomCoordinador") / f"PLABACOM_{date}_BD01" / "01 Cmg"
    
    csv_files = list(target_dir.glob("*.csv"))
    tsv_files = list(target_dir.glob("*.tsv"))
    
    if not csv_files or not tsv_files:
        print("Not found")
        continue
        
    df_cmg = pd.read_csv(csv_files[0], sep=";")
    df_fpen = pd.read_csv(tsv_files[0], sep=";")
    
    df_cmg = df_cmg[["nombre_barra", "nombre_barra_cmg", "tension"]].drop_duplicates()
    df_fpen = df_fpen[["BARRA_TRANSF", "BARRA_INFOTECNICA"]].drop_duplicates()
    df_fpen["BARRA_INFOTECNICA"] = df_fpen["BARRA_INFOTECNICA"].str.replace(r"\s*\[\.*\]\s*", "", regex=True)

    
    df_bi = pd.merge(left=df_cmg, right=df_fpen, left_on="nombre_barra_cmg", right_on="BARRA_TRANSF", how="left")
    df_bi = df_bi[["nombre_barra", "nombre_barra_cmg", "tension", "BARRA_INFOTECNICA"]]
    
    df_ntrac = df_bi[df_bi["BARRA_INFOTECNICA"].isna()] 
    df_trac = df_bi[~df_bi["BARRA_INFOTECNICA"].isna()] 
    
    df_ubi = pd.merge(left=df_be, right=df_trac, left_on="Nombre_BA", right_on="BARRA_INFOTECNICA", how="right")
    
    df_loc_temp = df_ubi[~df_ubi["Nombre_BA"].isna()] 
    df_nloc_temp = df_ubi[df_ubi["Nombre_BA"].isna()] 
    
    df_noloc_parcial = pd.concat([
        df_nloc_temp[["nombre_barra", "nombre_barra_cmg", "tension"]], 
        df_ntrac[["nombre_barra", "nombre_barra_cmg", "tension"]]
    ], ignore_index=True)
    
    lista_locs.append(df_loc_temp)
    lista_nolocs.append(df_noloc_parcial)

if lista_locs:
    df_locs = pd.concat(lista_locs, ignore_index=True)
    df_locs = df_locs.drop_duplicates(subset=["nombre_barra_cmg"]) 
else:
    df_locs = pd.DataFrame()

if lista_nolocs:
    df_nolocs = pd.concat(lista_nolocs, ignore_index=True)
    df_nolocs = df_nolocs.drop_duplicates(subset=["nombre_barra_cmg"])
else:
    df_nolocs = pd.DataFrame()

cantidad_rescatados = 0

if not df_locs.empty and not df_nolocs.empty:
    barras_rescatadas = df_nolocs[df_nolocs["nombre_barra_cmg"].isin(df_locs["nombre_barra_cmg"])]
    cantidad_rescatados = len(barras_rescatadas)
    df_nolocs = df_nolocs[~df_nolocs["nombre_barra_cmg"].isin(df_locs["nombre_barra_cmg"])]


print(f"Total barras conocidas finales: {len(df_locs)}")
print(f"Total incógnitas absolutas (nunca ubicadas): {len(df_nolocs)}")
print(f"Barras rescatadas (sin ubicación al inicio, ubicadas en meses posteriores): {cantidad_rescatados}")

Mirando la fecha  2505
Mirando la fecha  2506
Mirando la fecha  2507
Mirando la fecha  2508
Mirando la fecha  2509
Mirando la fecha  2510
Mirando la fecha  2511
Mirando la fecha  2512
Mirando la fecha  2601
Mirando la fecha  2602
Mirando la fecha  2603
Mirando la fecha  2604
Total barras conocidas finales: 1106
Total incógnitas absolutas (nunca ubicadas): 314
Barras rescatadas (sin ubicación al inicio, ubicadas en meses posteriores): 0


In [4]:
df_locs.to_parquet(Path(r"D:\ProyectoAnalisisElectrico\BarrasEstaciones\infraestructura_localizada.parquet"),  engine="pyarrow", compression="snappy")